# Task B-4: Model Versioning and Experimentation

## 1. Objective
To fulfill Task B-4, we compare three different versions of the Prometheus model to determine the optimal configuration for gym occupancy monitoring. We track hyperparameters, architectures, and performance metrics using **MLflow**.

### Models to Compare:
1. **Baseline_v1**: YOLOv10s (Small), imgsz=640, 50 epochs (The primary fine-tuned model).
2. **HighRes_v2**: YOLOv10s (Small), imgsz=800, 5 epochs (Testing high-resolution detail).
3. **QuickTest_v3**: YOLOv10n (Nano), imgsz=640, 5 epochs (Testing a lightweight architecture).

In [ ]:
from ultralytics import YOLO
import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import pathlib

# 1. Find Project Root
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    ROOT_DIR = os.path.abspath(os.path.join(cwd, '..'))
else:
    ROOT_DIR = cwd

print(f"Project Root: {ROOT_DIR}")

# 2. Fix MLflow Windows Path Issue
mlflow_path = pathlib.Path(ROOT_DIR) / "mlruns"
tracking_uri = mlflow_path.as_uri()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("Prometheus_Experimentation")

# 3. Prepare Absolute Path YAML
dataset_path = os.path.join(ROOT_DIR, 'model/data/final_dataset')
if not os.path.exists(dataset_path):
    dataset_path = os.path.join(ROOT_DIR, 'model/data/test_frames')

data_config = {
    'path': dataset_path,
    'train': 'images/train',
    'val': 'images/val',
    'names': { 0: 'person', 1: 'gym-machine' }
}

temp_yaml_path = os.path.join(ROOT_DIR, 'model/data/prometheus_experiment.yml')
with open(temp_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

# 4. Run QuickTest_v3 (if missing)
quick_test_results = os.path.join(ROOT_DIR, 'runs/detect/prometheus_runs/QuickTest_v3/results.csv')
if not os.path.exists(quick_test_results):
    print("Running QuickTest_v3...")
    model_n = YOLO('yolov10n.pt')
    model_n.train(data=temp_yaml_path, epochs=5, imgsz=640, batch=16, device='cpu', project='prometheus_runs', name='QuickTest_v3', exist_ok=True)
else:
    print("✅ QuickTest_v3 results found.")

Project Root: c:\Users\Phasit\Desktop\Prometheus\Prometheus
✅ QuickTest_v3 results found.


## 2. Quantitative Comparison Table
We extract the final metrics (mAP@0.5) from the `results.csv` files of each run.

In [ ]:
def get_metrics(run_name):
    csv_path = os.path.join(ROOT_DIR, f'runs/detect/prometheus_runs/{run_name}/results.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        final_row = df.iloc[-1]
        return {
            'Run Name': run_name,
            'mAP@0.5': round(final_row['metrics/mAP50(B)'], 4),
            'Precision': round(final_row['metrics/precision(B)'], 4),
            'Recall': round(final_row['metrics/recall(B)'], 4),
            'Training Time (s)': round(df['time'].sum(), 2)
        }
    return None

runs = ['baseline_model', 'HighRes_v2', 'QuickTest_v3']
comparison_data = []

for run in runs:
    m = get_metrics(run)
    if m: comparison_data.append(m)

comparison_df = pd.DataFrame(comparison_data)
comparison_df.set_index('Run Name', inplace=True)
display(comparison_df)

,mAP@0.5,Precision,Recall,Training Time (s)
Run Name,,,,
baseline_model,0.9870,0.9711,0.9583,2296.47
HighRes_v2,0.1719,0.3797,0.1667,44.76
QuickTest_v3,0.8020,0.7951,0.6723,1551.14


## 3. Justification & Final Selection

### Experimental Analysis:
1. **Baseline_v1 (Small, 640px, 50 Epochs)**:
   - **Result**: mAP@0.5 = **0.9870**.
   - **Insight**: This version achieved near-perfect accuracy on our validation set. The extended training (50 epochs) allowed the model to converge deeply on gym-specific geometry (weight stacks, pulley systems) which are often difficult to distinguish from background noise.

2. **HighRes_v2 (Small, 800px, 5 Epochs)**:
   - **Result**: mAP@0.5 = **0.1719**.
   - **Insight**: Despite the higher resolution, the performance was significantly lower after 5 epochs. This indicates that for this dataset, **resolution is less critical than training duration**. Increasing resolution also increased the training time per epoch without providing an immediate accuracy boost.

3. **QuickTest_v3 (Nano, 640px, 5 Epochs)**:
   - **Result**: mAP@0.5 = **0.8019**.
   - **Insight**: The Nano model performed impressively well, reaching 80% mAP in only 5 epochs. This suggests that the Prometheus dataset has clear features that even a tiny model can learn quickly. However, it still falls short of our 85% target in the short term.

### Final Selection: Baseline_v1
We select **Baseline_v1** as our production candidate. 
- **Evidence**: It is the only version to comfortably exceed the **85% mAP target**.
- **Reliability**: With a **95%+ recall**, it minimizes the risk of "ghost occupancy" or missing persons, which is critical for gym safety and management. 
- **Optimization**: While QuickTest_v3 is faster, the extra precision of the Small (s) model is necessary for high-accuracy occupancy counting in crowded gym environments.